opening for positional players:  https://lichess.org/study/0vNGLxJw/smynh89h   
opening dataset:https://huggingface.co/datasets/Lichess/chess-openings   

In [2]:
from huggingface_hub import notebook_login
from datasets import load_dataset
import pandas as pd
import numpy as np
import sqlite3
notebook_login()
#在输入中输入token

In [3]:
dataset=load_dataset("Lichess/chess-openings")
print(dataset['train'].features)
dataset.set_format(type="python", columns=["eco", "name", "pgn", "uci", "epd"])
openings_df = pd.DataFrame(dataset["train"]) 

{'eco-volume': Value('string'), 'eco': Value('string'), 'name': Value('string'), 'img': Image(mode=None, decode=True), 'pgn': Value('string'), 'uci': Value('string'), 'epd': Value('string')}


In [4]:
class TrieNode:
    def __init__(self):
        self.children={}
        self.opening_name=None

def build_tree(openings_df):
    root=TrieNode()
    for _,row in openings_df.iterrows():
        moves=row['uci'].split()
        node=root
        for move in moves:
            if move not in node.children:
                node.children[move]=TrieNode()
            node=node.children[move]
        node.opening_name=row['name']
    return root

def openings_identify(game_moves,trie_root):
    if not game_moves:
        return "Unknown"

    node=trie_root
    game_moves=game_moves.replace(',',' ')
    for move in game_moves.split():
        if move in node.children:
            node=node.children[move]
            if node.opening_name:
                return node.opening_name
        else:
            break
    return 'Unknown'
trie_root = build_tree(openings_df)
print(list(trie_root.children.keys())[:10])
db_file = r"C:\sqlite3\chess.db"
conn=sqlite3.connect(db_file)
games_df=pd.read_sql('SELECT * FROM games',conn)
tire_root=build_tree(openings_df)
games_df['Opening'] = games_df['Moves'].apply(lambda x: openings_identify(x, tire_root))
#映射字典
name_to_eco=dict(zip(openings_df['name'],openings_df['eco']))
games_df['ECO']=games_df['Opening'].map(name_to_eco)
print(games_df[['Moves', 'Opening','ECO']].head(50))
print(list(tire_root.children.keys())[:10])

['g1h3', 'e2e3', 'a2a3', 'f2f3', 'h2h3', 'g2g4', 'g2g3', 'h2h4', 'd2d3', 'b2b4']
                                                Moves            Opening  ECO
0   d2d4,g8f6,c2c4,e7e6,g1f3,b7b6,g2g3,c8b7,f1g2,f...  Queen's Pawn Game  D00
1   d2d4,d7d5,b1c3,c7c6,c1f4,g8f6,e2e3,c8f5,g1f3,e...  Queen's Pawn Game  D00
2   d2d4,g8f6,c2c4,c7c5,d4d5,b7b5,c4b5,a7a6,b5a6,e...  Queen's Pawn Game  D00
3   g1f3,d7d5,e2e3,b8c6,d2d4,c8f5,c2c4,e7e6,a2a3,a...  Zukertort Opening  A06
4   d2d4,g8f6,c2c4,e7e6,g2g3,f8b4,b1d2,c7c5,a2a3,b...  Queen's Pawn Game  D00
5   d2d4,d7d5,c2c4,c7c6,g1f3,g8f6,b1c3,d5c4,a2a4,c...  Queen's Pawn Game  D00
6                                                None            Unknown  NaN
7   e2e4,e7e6,d2d4,d7d5,b1d2,d5e4,d2e4,f8e7,g1f3,g...   King's Pawn Game  C20
8   e2e4,c7c5,b1c3,b8c6,g2g3,e7e6,f1g2,g7g6,d2d3,f...   King's Pawn Game  C20
9   e2e4,c7c5,g1f3,e7e6,g2g3,b8c6,f1g2,g8f6,d1e2,d...   King's Pawn Game  C20
10  e2e4,c7c5,g1f3,e7e6,d2d4,c5d4,f3d4,g8f6,b1c3,b...   King'

In [5]:
games_df.shape
games_df.head()

,uid,Event,Site,Date,Round,White,Black,Result,WhiteElo,BlackElo,TimeControl,EndTime,Termination,Moves,Opening,ECO
0,1,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Petra Kejzar,Daniel Barria,0-1,2177,2637,180+1,15:01:06 GMT+0000,manitodeplomo胜，对手认输,"d2d4,g8f6,c2c4,e7e6,g1f3,b7b6,g2g3,c8b7,f1g2,f...",Queen's Pawn Game,D00
1,2,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Nataliya Buksa,Jorge Carlos Antonio,1-0,2639,2195,180+1,15:02:10 GMT+0000,Natalya_Buksa胜，对手认输,"d2d4,d7d5,b1c3,c7c6,c1f4,g8f6,e2e3,c8f5,g1f3,e...",Queen's Pawn Game,D00
2,3,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Hubert Zieba,Nikolaos Skiadopoulos,1-0,2607,2049,180+1,15:02:41 GMT+0000,Chomiczek786胜，对手认输,"d2d4,g8f6,c2c4,c7c5,d4d5,b7b5,c4b5,a7a6,b5a6,e...",Queen's Pawn Game,D00
3,4,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Karina Ambartsumova,Nino Maisuradze,1-0,2709,2358,180+1,15:02:42 GMT+0000,karinachess1胜，对手认输,"g1f3,d7d5,e2e3,b8c6,d2d4,c8f5,c2c4,e7e6,a2a3,a...",Zukertort Opening,A06
4,5,Early-Titled-Tuesday-Blitz-April-01-2025,Chess.com,2025.04.01,1,Alexander Donchenko,Andriy Diachek,1-0,2835,2494,180+1,15:02:49 GMT+0000,Alexander_Donchenko胜，对手认输,"d2d4,g8f6,c2c4,e7e6,g2g3,f8b4,b1d2,c7c5,a2a3,b...",Queen's Pawn Game,D00


opening fre

In [6]:
opening_counts=games_df['Opening'].value_counts()
print(opening_counts.head(30))

Opening
King's Pawn Game        473079
Queen's Pawn Game       338469
Zukertort Opening        99613
English Opening          55018
Nimzo-Larsen Attack      15762
Hungarian Opening         6704
Unknown                   5466
Bird Opening              5153
Van Geet Opening          3670
Anderssen's Opening       3131
Grob Opening              2187
Polish Opening            2027
Mieses Opening            1696
Van't Kruijs Opening      1523
Kádas Opening             1520
Saragossa Opening          746
Clemenz Opening            722
Ware Opening               463
Barnes Opening             335
Sodium Attack              236
Amar Opening               188
Name: count, dtype: int64


In [17]:
# 把结果转成数值：1=白胜, 0.5=平, 0=黑胜
def result_to_score(result, is_white=True):
    if result == '1-0':
        return 1 if is_white else 0
    elif result == '0-1':
        return 0 if is_white else 1
    else:
        return 0.5

games_df['WhiteScore'] = games_df['Result'].apply(lambda x: result_to_score(x, True))
games_df['BlackScore'] = games_df['Result'].apply(lambda x: result_to_score(x, False))

# 统计开局平均胜率（白方）
opening_winrate_white = games_df.groupby('Opening')['WhiteScore'].mean()
print(opening_winrate_white.sort_values(ascending=False))
print('----------------------------------------')
opening_winrate_black = games_df.groupby('Opening')['BlackScore'].mean()
print(opening_winrate_black.sort_values(ascending=False))
print('----------------------------------------')

opening_1 = games_df.groupby('Opening').agg(
    WhiteWinRate=('WhiteScore', 'mean'),
    BlackWinRate=('BlackScore', 'mean'),
    GamesCount=('Result', 'count')  # 可选：统计每个开局的局数
).reset_index()

# 按白方胜率排序
opening_1 = opening_1.sort_values(by='WhiteWinRate', ascending=False)

print(opening_1)

Opening
Amar Opening            0.654255
Ware Opening            0.644708
Clemenz Opening         0.635734
Anderssen's Opening     0.626956
Sodium Attack           0.625000
Kádas Opening           0.610526
Van Geet Opening        0.590736
Nimzo-Larsen Attack     0.586918
Polish Opening          0.582634
Saragossa Opening       0.567694
Grob Opening            0.563786
Zukertort Opening       0.557096
Hungarian Opening       0.552357
Bird Opening            0.544246
English Opening         0.539605
Van't Kruijs Opening    0.531845
Queen's Pawn Game       0.528264
Mieses Opening          0.526238
King's Pawn Game        0.520033
Barnes Opening          0.510448
Unknown                 0.032656
Name: WhiteScore, dtype: float64
----------------------------------------
Opening
Unknown                 0.967344
Barnes Opening          0.489552
King's Pawn Game        0.479967
Mieses Opening          0.473762
Queen's Pawn Game       0.471736
Van't Kruijs Opening    0.468155
English Opening    

In [32]:
# Unknown 开局，具体走子全是None， 而且绝大多数是黑的赢了
unknown_games = games_df[games_df['Opening'] == 'Unknown']
none_moves_games = games_df[games_df['Moves'].isna()]
print(f"Moves为None的总局数: \n{len(none_moves_games)}\n----------------------------------------")
print(f"Moves为None的局数的结果分布：\n{none_moves_games['Result'].value_counts()}\n----------------------------------------")
#居然几乎都是白方超时！
print(f"Moves=None 局的 Termination 分布：\n{none_moves_games['Termination'].str[-4:].value_counts()}")

Moves为None的总局数: 
5466
----------------------------------------
Moves为None的局数的结果分布：
Result
0-1        5256
1-0         147
1/2-1/2      63
Name: count, dtype: int64
----------------------------------------
Moves=None 局的 Termination 分布：
Termination
对手超时    4827
对手认输     576
协议和棋      63
Name: count, dtype: int64


opening choices for different level players

In [ ]:
elo_opening = games_df.groupby('Opening')[['WhiteElo', 'BlackElo']].mean()
print(elo_opening.sort_values('WhiteElo', ascending=False).head(10))

                        WhiteElo     BlackElo
Opening                                      
Amar Opening         2696.276596  2517.271277
Sodium Attack        2695.750000  2509.864407
Ware Opening         2679.419006  2468.185745
Anderssen's Opening  2666.717981  2535.031939
Clemenz Opening      2652.547091  2561.288089
Kádas Opening        2619.123684  2536.164474
Van Geet Opening     2616.871662  2543.635422
Nimzo-Larsen Attack  2615.433257  2541.929958
Zukertort Opening    2592.138626  2557.609900
Polish Opening       2570.199309  2523.367538
